# NumPy: arrays, axes, broadcasting

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~50 min

## Creating and reshaping arrays

Everything in NumPy is an `ndarray`: one block of memory holding same-typed numbers. Build one from a list with `np.array`, or let NumPy fill it: `np.zeros`, `np.ones`, `np.arange` (like `range`), `np.linspace` (n evenly spaced points, ends included), `np.random.randn` (standard normal). `.dtype` says what each cell holds; `.shape` says how the cells are arranged.



In [ ]:
import numpy as np

np.array([1, 2, 3])               # dtype int64 — from a Python list
np.zeros((2, 3))                  # 2×3 of float64 zeros
np.arange(0, 10, 2)               # [0 2 4 6 8] — like range, but an array
np.linspace(0, 1, 5)              # [0. 0.25 0.5 0.75 1.] — 5 points, ends INCLUDED
np.random.randn(3)                # 3 standard-normal samples

`.reshape(r, c)` rewraps the same numbers into a new shape — it copies nothing and reorders nothing. `-1` means "work this dimension out yourself". `.T` transposes (swaps axes); `.ravel()` flattens to 1-D:



In [ ]:
a = np.arange(12)              # [0..11], shape (12,)
a.reshape(3, 4)                  # 3 rows × 4 columns
a.reshape(3, -1)                 # same thing — -1 infers the 4
a.reshape(3, 4).T.shape          # (4, 3)
a.reshape(2, 2, 3).ravel()       # back to (12,)

- `np.array([[1, 2], [3]])` fails or makes an object array: rows must have equal length — an ndarray is a box, not a list of lists.
- `np.arange(0, 1, 0.1)` can surprise you (0.1 isn't exact in binary); when you know the point count, `linspace` is safer.
- float64 is the default and costs 8 bytes per number; images and neural nets usually want float32 — pass `dtype="float32"` at creation.



## Axes and shape algebra

`ndarray.shape` is a tuple of dimension sizes: a table with 3 rows and 4 columns has shape `(3, 4)`; axis 0 runs **down** the rows, axis 1 runs **across** the columns. `arr.mean(axis=k)` collapses dimension k — "average along that direction" — and removes it from the shape. `axis=0` on the table gives 4 column means, `axis=1` gives 3 row means. `keepdims=True` leaves a size-1 axis in its place instead of deleting it.

```
(32, 64, 64, 3).mean(axis=0)  →  (64, 64, 3)   (batch of images → per-pixel mean)
```

```
(3, 4).mean(axis=1, keepdims=True)  →  (3, 1)   (reusable in broadcasting)
```

IOAI (the International Olympiad in Artificial Intelligence) relevance: metrics, normalization and aggregation are all axis reductions — per-sample losses reduce over the feature axis, per-class statistics over the sample axis, per-channel means over the batch axis. A wrong axis is the most common silent bug in contest notebooks: the code runs, the numbers are wrong.



## Vectorized reductions



In [ ]:
import numpy as np

X = np.random.randn(1000, 5)              # 1000 random samples, 5 features (randn draws standard-normals)

mu = X.mean(axis=0)                       # (5,) — per-feature mean
sd = X.std(axis=0) + 1e-8                 # (5,) — epsilon guards zero variance
X_std = (X - mu) / sd                     # (1000, 5) — "z-score": (value − mean) / std per column

row_sums = X.sum(axis=1, keepdims=True)   # (1000, 1)
X_norm = X / row_sums                     # each row sums to 1

**Execution pitfalls**

- `X.sum(axis=1)` has shape `(1000,)`, and `X / X.sum(axis=1)` raises `ValueError` — NumPy cannot line a length-1000 vector up with the 5 columns (the rule is explained in the next step). `keepdims=True` keeps the shape `(1000, 1)`, one number per row, which divides cleanly.
- `X[mask]` picks the rows where a boolean mask is True; `.mean()` on an empty selection returns `nan` (not-a-number) with a RuntimeWarning — check `mask.any()` first, or the NaN contaminates every downstream feature.
- `sd == 0` on a constant column → `inf`/`nan`. Add epsilon before dividing, never after.



### ✏️ X has shape (100, 3): 100 samples, 3 features. What does X.mean(axis=1) return?

**A.** The global mean — a scalar
**B.** An error: axis out of bounds
**C.** The mean of each sample — shape (100,)
**D.** The mean of each feature — shape (3,)

<details><summary>Answer</summary>

**C.** The mean of each sample — shape (100,)

axis=1 reduces across the 3 features, leaving one value per sample. Per-feature mean is axis=0.

</details>

## Broadcasting rules

Binary ops align shapes **from the right**. For each dimension pair: the sizes must be equal, or one must be 1 (it stretches), or one must be missing (it becomes 1, then stretches). Anything else is a `ValueError`.

Try it on the figure's example, `(3,1) + (1,4)`: right-align the shapes and check each pair — 3 vs 1, the 1 stretches to 3; 1 vs 4, the 1 stretches to 4 — result `(3,4)`. The failure mode: `(6,) + (7,8)` right-aligns the 6 under the 8, and 6 ≠ 8 with no 1 to stretch → `ValueError`. A missing dim counts as 1: `(3,4) + (4,)` reads the vector as `(1,4)` and stretches it over all 3 rows.

> 🖼 *Broadcasting: a (3,1) column and a (1,4) row stretch into a (3,4) grid — no loops, no copies.*



In [ ]:
import numpy as np      # shapes, not values — check .shape of each result
(np.zeros((8, 1, 6)) + np.zeros((7, 1))).shape   # (8, 7, 6)
(np.zeros((3, 4)) - np.zeros((4,))).shape        # (3, 4)  row vector applied to every row
(np.zeros((3, 4)) - np.zeros((3, 1))).shape      # (3, 4)  column applied to every column
(np.zeros((3,)) + np.zeros((3, 1))).shape        # (3, 3)  every element meets every element — a common accident

You insert a size-1 axis with `None` (an alias for `np.newaxis`) inside the brackets. For `v = np.array([1, 2, 3])`: `v[:, None]` is a column of shape `(3, 1)`, `v[None, :]` is a row of shape `(1, 3)`. Subtracting them gives every pair of differences at once:



In [ ]:
v = np.array([1, 2, 3])
v[:, None] - v[None, :]
# [[ 0, -1, -2],
#  [ 1,  0, -1],
#  [ 2,  1,  0]]        shape (3, 3): row i, column j holds v[i] − v[j]

That one idea — add an axis, let broadcasting pair everything with everything — replaces most double loops in contest code.

IOAI relevance: feature standardization, distance matrices for k-NN and embedding matching, one-hot masks, scoring every sample against class prototypes — all broadcasting, zero Python loops.



## Broadcast patterns that replace loops

Pairwise distances between all rows of `A` (n points, d coordinates each) — the same trick as above, one axis up:



In [ ]:
A = np.random.randn(100, 3)             # 100 points in 3-D
diff = A[:, None, :] - A[None, :, :]     # (100, 1, 3) − (1, 100, 3) → (100, 100, 3): every pair
D = np.sqrt((diff ** 2).sum(axis=-1))    # axis=-1 = the last axis (the 3 coordinates) → (100, 100)

**Softmax** turns a row of raw scores (called **logits**) into probabilities: `exp(z) / exp(z).sum()`. Subtracting the row maximum first does not change the result (it cancels in the ratio) but keeps `exp` from overflowing:



In [ ]:
logits = np.random.randn(1000, 10)                   # 1000 samples, 10 class scores
z = logits - logits.max(axis=1, keepdims=True)       # (1000, 10) − (1000, 1): largest score becomes 0
e = np.exp(z)
p = e / e.sum(axis=1, keepdims=True)                 # each row sums to 1

**Execution pitfalls**

- `(n,)` vs `(n, 1)`: subtracting two `(n,)` vectors gives `(n,)`; make one `(n, 1)` and you silently get an `(n, n)` outer result. Print `.shape` before trusting the result.
- `A[:, None, :] - A[None, :, :]` allocates `n² · d` numbers — at n = 10 000 points with d = 128 that is 12.8 billion float64s, over 100 GB. Process the rows in chunks of ≈ 1000, or use `scipy.spatial.distance.cdist(A, A)`, which does the chunking for you.
- `np.exp(x)` is infinite for x above about 709 — beyond that, exp itself overflows the float64 range. Without the max-subtraction one large logit gives `inf / inf = nan` for the whole row.



### ✏️ A has shape (8, 1, 6) and B has shape (7, 1). What is the shape of A + B?

**A.** (8, 7, 6)
**B.** (8, 7, 1)
**C.** ValueError: operands could not be broadcast
**D.** (8, 1, 6)

<details><summary>Answer</summary>

**A.** (8, 7, 6)

Right-align: 6 vs 1 → 6, 1 vs 7 → 7 (the 1 stretches), 8 vs missing → 8. Result (8, 7, 6).

</details>

## Boolean and fancy indexing

`a[mask]` with a boolean array selects elements where the mask is True. `a[idx]` with an integer array gathers elements — the output shape follows `idx`, not `a`. Both return **copies**: the result is a new array, so changing it does not change `a` (plain slices like `a[2:5]` are **views** that share memory).

`np.argsort(scores)` returns the indices that would sort `scores` ascending — `argsort([0.2, 0.9, 0.5])` is `[0, 2, 1]`. Sort `-scores` to get descending order, then take the first k indices for the top-k.

IOAI relevance: thresholding probabilities into predictions, masking corrupt pixels or frames, top-k selection for retrieval metrics, dropping outliers before fitting.



## Masks and top-k



In [ ]:
prob = np.random.rand(len(X))              # one score per row of X
scores = np.random.rand(len(X), 4)       # 4 candidate scores per row

pred = prob > 0.5                        # boolean mask — one flag per row
X_clean = X[~np.isnan(X).any(axis=1)]    # drop rows containing NaN
X[pred] = 0                              # write through the mask (pred has one flag per row)

top3 = np.argsort(-scores, axis=1)[:, :3]  # top-3 column indices per row
label = np.where(prob > 0.5, 1, 0)         # element-wise select

**Execution pitfalls**

- Combine masks with `&`, `|`, `~` — never `and`/`or`/`not` (they raise `ValueError: truth value of an array is ambiguous`). Parenthesize each condition: `(a > 0) & (b < 1)`.
- `a[idx] = v` mutates `a`, but `b = a[idx]; b[0] = 9` mutates a copy — `a` is unchanged. Chained `a[mask][:, 2] = v` silently does nothing.
- `argsort` returns indices that would sort the array; negate the scores for descending order. `[:, :k]` off-by-one is the classic top-k bug.

> **Note.** Train this in the **Tensor Puzzles** tab: 21 puzzles where you rebuild NumPy ops (sum, cumsum, bincount…) in one line using only broadcasting, indexing and `where`.



### ✏️ p = np.array([0.7, 0.2, 0.9]); p[p > 0.5] = 1. What is p now?

**A.** ValueError: cannot assign
**B.** [1, 0, 1]
**C.** [1.0, 0.2, 1.0]
**D.** [1.0, 0.0, 1.0]

<details><summary>Answer</summary>

**C.** [1.0, 0.2, 1.0]

The mask selects positions 0 and 2; assignment writes through it. The masked-out element keeps its value.

</details>

---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Min-max scale columns

●○○ · ~8 min

> Min-max scaling maps every column to [0, 1]: (x − min) / (max − min). Constant columns need a guard: max − min is 0, so the division produces NaN. The convention here is to map them to 0.

Implement `minmax(X)` for a 2-D matrix: scale each **column** to [0, 1]. If a column is constant, return zeros for it.

<details><summary>Hint</summary>

Use axis=0 reductions with keepdims=True for broadcasting. Guard with np.where on (max − min) == 0 — or better, compute the denom with np.where(denom == 0, 1, denom) so you never divide by zero.

</details>


In [ ]:
import numpy as np

def minmax(X):
    X = np.asarray(X, dtype=float)
    # your code
    return X

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    X = rng.integers(-10, 11, size=(int(rng.integers(4, 9)), int(rng.integers(2, 6)))).astype(float)\n    if rng.random() < 0.4:\n        X[:, int(rng.integers(0, X.shape[1]))] = 3.0   # constant column\n    return {\"X\": X}\n\ndef minmax(X):\n    X = np.asarray(X, dtype=float)\n    lo, hi = X.min(0, keepdims=True), X.max(0, keepdims=True)\n    d = hi - lo\n    return np.where(d == 0, 0.0, (X - lo) / np.where(d == 0, 1.0, d))", "minmax", 20)

### Task 2 — Label smoothing of one-hot targets

●○○ · ~8 min

> Label smoothing replaces a hard one-hot target (1 on the true class, 0 elsewhere) with a soft target: the true class gets 1 − eps + eps/k and every other class gets eps/k. It stops a classifier from driving its logits to infinity to satisfy an overconfident target, which makes it a simple and effective regularizer for cross-entropy training. With eps = 0 the formula collapses back to ordinary one-hot — a good sanity check.

Implement `label_smooth(y, k, eps)` → an (n, k) float matrix where row i has `1 - eps + eps/k` at position `y[i]` and `eps/k` everywhere else.

```
y=[1], k=3, eps=0.3 → [[0.1, 0.8, 0.1]]
```

<details><summary>Hint</summary>

Start from np.eye(k)[y] for the hard part, then blend: (1 - eps) * onehot + eps / k.

</details>


In [ ]:
import numpy as np

def label_smooth(y, k, eps):
    y = np.asarray(y)
    # your code
    return np.zeros((len(y), k))

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n, k = int(rng.integers(3, 30)), int(rng.integers(2, 7))\n    return {\"y\": rng.integers(0, k - 1, n) if rng.random() < 0.35 else rng.integers(0, k, n),\n            \"k\": k,\n            \"eps\": float(rng.choice([0.0, 0.05, 0.1, 0.2, 0.3]))}\n\ndef label_smooth(y, k, eps):\n    y = np.asarray(y)\n    return (1.0 - eps) * np.eye(k)[y] + eps / k", "label_smooth", 20)

### Task 3 — Top-k indices by score

●●○ · ~10 min

> Retrieval and ranking tasks (semantic search, KNN, RAG) all end with 'take the k highest scores'. `np.argsort(-s)[:k]` is the idiom — know it cold, including the sign flip for descending order.

Implement `topk(scores, k)` → indices of the `k` largest scores, in decreasing order. Ties may be returned in any order.

<details><summary>Hint</summary>

np.argsort sorts ascending; negate the scores, or sort then [::-1].

</details>


In [ ]:
import numpy as np

def topk(scores, k):
    # your code
    return []

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(5, 30))\n    return {\"scores\": rng.normal(0, 1, n), \"k\": int(rng.integers(1, n))}\n\ndef topk(scores, k):\n    return np.argsort(-np.asarray(scores))[:k]", "topk", 20)

---

**Go deeper**
- [NumPy: broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [CS231n: Python & NumPy tutorial](https://cs231n.github.io/python-numpy-tutorial/)